# 网络

## 简介

本教程概述了 **skrf** 的微波网络分析功能。在本教程以及 scikit-rf 文档的其余部分中，我们假定已将 **skrf** 导入为 `rf`。是否在您自己的代码中遵循此约定由您决定。

In [ ]:
import numpy as np

import skrf as rf

如果出现导入错误，请参阅[安装指南](Installation.ipynb)。

## 创建网络对象

**skrf** 提供了用于表示 N 端口微波[网络](../api/network.rst)的对象。可以通过多种方式创建[网络](../api/network.rst)：- 从 Touchstone 文件- 从散射参数- 从阻抗参数- 从其他射频参数（Y、ABCD、T 等）以下是一些针对每种情况的示例。

### 从 Touchstone 文件创建网络[Touchstone 文件](https://en.wikipedia.org/wiki/Touchstone_file)（`.sNp` 文件，其中 `N` 是端口数）是一种事实上的标准，用于导出 N 端口网络参数数据和线性有源器件、无源滤波器、无源器件或互连网络的噪声数据。从 Touchstone 文件创建网络非常简单：

In [ ]:
from skrf import Frequency, Network

ring_slot = Network('data/ring slot.s2p')

请注意，一些软件，如 ANSYS HFSS，会在 Touchstone 标准中添加额外信息，例如注释、仿真参数、端口阻抗或 Gamma（波数）。如果检测到这些数据，它们也会被导入。

如果将网络参数输入到命令行，将会打印出该网络的简短描述。

In [ ]:
ring_slot

### 从散射参数创建网络网络也可以通过直接传递 `frequency`、`s` 散射参数（以及可选的端口阻抗 `z0`）的值来创建。一个 N 端口网络的散射矩阵应该是一个 Numpy 数组，其形状为 `(nb_f, N, N)`，其中 `nb_f` 是频率点的数量，`N` 是网络的端口数量。<img src="figures/arrays_s_vs_f.svg" width="300">

In [ ]:
# dummy 2-port network from Frequency and s-parameters
freq = Frequency(1, 10, 101, 'ghz')
rng = np.random.default_rng()
s = rng.uniform(size=(101, 2, 2)) + 1j*rng.uniform(size=(101, 2, 2))  # random complex numbers
# if not passed, will assume z0=50. name is optional but it's a good practice.
ntwk = Network(frequency=freq, s=s, name='random values 2-port')
ntwk

In [ ]:
ntwk.plot_s_db()

通常，散射参数存储在单独的数组中。在这种情况下，需要构建 s 矩阵：

In [ ]:
# let's assume we have separate arrays for the frequency and s-parameters
f = np.array([1, 2, 3, 4]) # in GHz
S11 = rng.uniform(size=4)
S12 = rng.uniform(size=4)
S21 = rng.uniform(size=4)
S22 = rng.uniform(size=4)

# Before creating the scikit-rf Network object, one must forge the Frequency and S-matrix:
freq2 = rf.Frequency.from_f(f, unit='GHz')

# forging S-matrix as shape (nb_f, 2, 2)
# there is probably smarter way, but less explicit for the purpose of this example:
s = np.zeros((len(f), 2, 2), dtype=complex)
s[:,0,0] = S11
s[:,0,1] = S12
s[:,1,0] = S21
s[:,1,1] = S22

# constructing Network object
ntw = rf.Network(frequency=freq2, s=s)

print(ntw)

如果需要，特性阻抗可以作为标量（所有频率相同）、列表或数组传递：

In [ ]:
ntw2 = rf.Network(frequency=freq, s=s, z0=25, name='same z0 for all ports')
print(ntw2)
ntw3 = rf.Network(frequency=freq, s=s, z0=[20, 30], name='different z0 for each port')
print(ntw3)
ntw4 = rf.Network(frequency=freq, s=s, z0=rng.uniform(size=(4,2)), name='different z0 for each frequencies and ports')
print(ntw4)

### 从 Z 参数推导

由于网络也可以通过其阻抗参数来定义，因此 `skrf.Network` 类中有一个 `from_z()` 方法：

In [ ]:
# 1-port network example
z = np.full((len(freq), 1, 1), 10j)  # replicate z=10j for all frequencies

ntw = rf.Network(frequency=freq, z=z)
print(ntw)

### 从其他网络参数：Y、ABCD、H、T也可以从其他类型的射频参数生成网络：[Y](https://en.wikipedia.org/wiki/Two-port_network#Admittance_parameters_(y-parameters))、[ABCD](https://en.wikipedia.org/wiki/Two-port_network#ABCD-parameters)、[H](https://en.wikipedia.org/wiki/Two-port_network#Hybrid_parameters_(h-parameters)) 或 [T](https://en.wikipedia.org/wiki/Two-port_network#Scattering_transfer_parameters_(T-parameters))。在创建 `Network` 时，分别使用 `y=`、`a=`、`h=` 或 `t=` 参数。例如，串联阻抗的 [ABCD 参数](https://en.wikipedia.org/wiki/Two-port_network#ABCD-parameters) 为：$$\left[\begin{array}{cc}1 & Z \\0 & 1\end{array}\right]$$因此，可以这样创建相应的 `Network`：

In [ ]:
z = 20
abcd = np.array([[1, z],
              [0, 1]])

a = np.tile(abcd, (len(freq),1,1))
ntw = Network(frequency=freq, a=a)
print(ntw)

请注意，如果需要，还提供了一些便捷函数，用于将某些参数转换为其他参数：| 从\到 |   S   |   Z   |   Y   |  ABCD |   T   |   H   ||:-------:|:-----:|:-----:|:-----:|:-----:|:-----:|:-----:||    S    |       | `s2z` | `s2y` | `s2a` | `s2t` | `s2h` ||    Z    | `z2s` |       | `z2y` | `z2a` | `z2t` | `z2h` ||    Y    | `y2s` | `y2z` |       |       | `y2t` |       ||   ABCD  | `a2s` | `a2z` |       |       |       |       ||    T    | `t2s` | `t2z` | `t2y` |       |       |       ||    H    | `h2s` | `h2z` |       |       |       |       |

In [ ]:
# example: converting a -> s
s = rf.network.a2s(a)
# checking that these S-params are the same
np.all(ntw.s == s)

## 基本属性微波[网络](../api/network.rst)的基本属性由以下属性提供：* `Network.s`：散射参数矩阵。* `Network.z0`：端口特性阻抗矩阵。* `Network.frequency`：频率对象。

[Network](../api/network.rst) 对象具有许多其他属性和方法。如果您使用的是 IPython，那么这些属性和方法可以在命令行中通过“Tab”键自动补全。

```In [1]: ring_slot.sring_slot.line.s              ring_slot.s_arcl         ring_slot.s_imring_slot.line.s11            ring_slot.s_arcl_unwrap  ring_slot.s_mag...```

所有网络参数在内部都表示为复数 `numpy.ndarray`。散射参数的形状为 `(nfreq, nport, nport)`：

In [ ]:
np.shape(ring_slot.s)

## 切片

您可以按照您想要的方式对 `Network.s` 属性进行切片。

In [ ]:
ring_slot.s[:11,1,0]  # get first 10 values of S21

也可以直接对 Network 对象进行频率分片，如下所示：

In [ ]:
ring_slot[0:10] #  Network for the first 10 frequency points

或者使用对用户友好的字符串，

In [ ]:
ring_slot['80-90ghz']

请注意，直接对 skrf.Network 对象进行切片操作会返回一个新的 skrf.Network 对象。因此，一种简洁的方式来表示二维切片是：

In [ ]:
ring_slot.s11['80-90ghz']

## 绘图

除了其他功能之外，[Network](../api/network.rst) 类的方法还提供了方便的方式来绘制网络参数的各个分量，例如：* `Network.plot_s_db`：以对数刻度绘制散射参数的幅值。* `Network.plot_s_deg`：以度为单位绘制散射参数的相位。* `Network.plot_s_smith`：在史密斯图上绘制复数散射参数。* ...如果您想使用 skrf 的绘图样式，

In [ ]:
%matplotlib inline
rf.stylely()

要在史密斯图上绘制 `ring_slot` 的所有四个散射参数。

In [ ]:
ring_slot.plot_s_smith()

将此与切片功能结合起来，

In [ ]:
from matplotlib import pyplot as plt

plt.title('Ring Slot $S_{21}$')

ring_slot.s11.plot_s_db(label='Full Band Response')
ring_slot.s11['82-90ghz'].plot_s_db(lw=3,label='Band of Interest')

有关绘图的更多详细信息，请参阅[绘图](Plotting.ipynb)。

## 算术运算可以通过重载的运算符对散射参数矩阵执行逐元素数学运算。为了说明其用法，请加载存储在 `data` 模块中的几个 `Network` 对象。

In [ ]:
from skrf.data import wr2p2_delayshort as delayshort
from skrf.data import wr2p2_short as short

short - delayshort
short + delayshort
short * delayshort
short / delayshort


所有这些操作都返回 [Network](../api/network.rst) 类型。例如，要绘制 `short` 和 `delay_short` 之间的复数差，可以这样做：

In [ ]:
difference = (short - delayshort)
difference.plot_s_mag(label='Mag of difference')

另一个常见应用是使用除法运算符来计算相位差。

In [ ]:
(delayshort/short).plot_s_deg(label='Detrended Phase')

线性算子也可以与标量或长度与 [Network](../api/network.rst) 相同的 `numpy.ndarray` 一起使用。

In [ ]:
hopen = (short*-1)
hopen.s[:3,...]

In [ ]:
rando =  hopen *rng.uniform(size=len(hopen))
rando.s[:3,...]

## 网络比较比较运算符也可用于网络：

In [ ]:
short == delayshort

In [ ]:
short != delayshort

## 级联和去除也可以通过运算符对 2 端口网络进行级联和去除。可以通过幂运算符 `**` 调用 `cascade` 函数，以计算一个新的网络，该网络是两个单独网络 `line` 和 `short` 的级联连接。

In [ ]:
short = rf.data.wr2p2_short
line = rf.data.wr2p2_line
delayshort = line ** short

可以通过级联网络的*逆*来实现去嵌入。可以通过属性 `Network.inv` 访问网络的逆。要从 `delay_short` 中去除 `short`，

In [ ]:
short_2 = line.inv ** delayshort

short_2 == short

级联运算符同样适用于 2N 端口网络。这在[关于平衡网络的示例](https://github.com/scikit-rf/scikit-rf/blob/master/examples/networktheory/Balanced%20Network%20De-embedding.ipynb) 中有所说明。例如，假设您想级联三个 4 端口网络 `ntw1`、`ntw2` 和 `ntw3`，您可以使用：`resulting_ntw = ntw1 ** ntw2 ** ntw3`。请注意，`**` 级联运算符在 4 端口网络中使用的端口方案为：

```ntw1    ntw2    ntw3+----+  +----+  +----+0-|0  2|--|0  2|--|0  2|-21-|1  3|--|1  3|--|1  3|-3+----+  +----+  +----+```

关于连接网络的更多文档请参见此处：[连接网络](./Connecting_Networks.ipynb)。

## 连接多端口**skrf** 支持连接 N 端口网络中的任意端口。它通过一种名为子网络增长的算法来实现这一点[[1]](#References)，该算法可通过函数 `connect()` 使用。例如，可以通过以下方式连接理想 3 路分路器的其中一个端口：

In [ ]:
tee = rf.data.tee
tee

将 T 型接头的端口 `1` 连接到延迟短路的端口 `0`。

In [ ]:
terminated_tee = rf.network.connect(tee, 1, delayshort, 0)
terminated_tee

请注意，此函数会考虑端口阻抗。如果两个连接的端口具有不同的端口阻抗，则会插入适当的阻抗失配。有关连接网络（Networks）的更多信息，请参见此处：[连接网络（Connecting Networks）](./Connecting_Networks.ipynb)。对于多个任意 N 端口网络之间的更高级连接，`Circuit` 对象更为适用，因为它允许显式定义端口和网络之间的连接。有关更多信息，请参阅[电路文档（Circuit documentation）](Circuit.ipynb)。

## 插值与连接一个常见的需求是改变 [Network](../api/network.rst) 中频率点的数量。为了使用运算符和级联函数，所涉及的网络必须具有匹配的频率，例如。如果两个网络具有不同的频率信息，则会引发错误。

In [ ]:
from skrf.data import wr2p2_line1 as line1

line1

line1+line---------------------------------------------------------------------------IndexError                                Traceback (最近的调用)<ipython-input-49-82040f7eab08> in <module>()----> 1 line1+line/home/alex/code/scikit-rf/skrf/network.py in __add__(self, other)    500     501     if isinstance(other, Network):--> 502         self.__compatible_for_scalar_operation_test(other)    503         result.s = self.s + other.s    504     else:/home/alex/code/scikit-rf/skrf/network.py in __compatible_for_scalar_operation_test(self, other)    701     '''    702     if other.frequency  != self.frequency:--> 703         raise IndexError('网络必须具有相同的频率。请参阅 `Network.interpolate`')    704     705     if other.s.shape != self.s.shape:IndexError: 网络必须具有相同的频率。请参阅 `Network.interpolate`

可以使用 `Network.resample` 函数，通过在频率轴上对其中一个 `skrf.Network` 对象进行插值来解决此问题。

In [ ]:
line1.resample(201)
line1

现在我们可以开始做一些事情了。

In [ ]:
line1 + line

您也可以从 `Frequency` 对象中进行插值。例如：

In [ ]:
line.interpolate(line1.frequency)

一个相关的应用是需要将覆盖不同频率范围的网络组合起来。可以使用 `stitch` 函数将两个网络连接（或拼接）在一起，从而沿其频率轴组合网络。例如，可以将一个 WR-2.2 网络与一个 WR-1.5 网络组合起来。

In [ ]:
from skrf.data import wr1p5_line, wr2p2_line

big_line = rf.network.stitch(wr2p2_line, wr1p5_line)
big_line

## 读取和写入为了长期存储数据，**skrf** 支持读取 [Touchstone 文件格式](http://en.wikipedia.org/wiki/Touchstone_file)，并部分支持写入。读取操作通过上面所示的 `Network` 初始化器完成，写入操作通过 `Network.write_touchstone()` 方法完成。对于**临时**数据存储，**skrf** 对象可以使用 `skrf.io.general.read` 和 `skrf.io.general.write` 函数进行 [序列化](http://docs.python.org/2/library/pickle.html)。选择使用临时序列化而非 Touchstone 文件的原因是，序列化可以存储网络的所有属性，而 Touchstone 文件仅存储部分信息。

In [ ]:
rf.io.write('data/myline.ntwk',line) # write out Network using pickle

In [ ]:
ntwk = Network('data/myline.ntwk') # read Network using pickle

通常，需要分析的文件的数量很多，它们可能都存储在一个目录下。`rf.read_all` 可以快速地从目录中的所有文件中创建 `Network` 对象。例如，要加载 `data/` 目录中所有包含字符串 `'wr2p2'` 的 **skrf** 文件。

In [ ]:
dict_o_ntwks = rf.io.read_all(rf.data.pwd, contains = 'wr2p2')
dict_o_ntwks

有时，您会知道需要分析的文件列表。`rf.read_all` 还可以接受一个 files 参数。此示例文件列表仅包含同一目录中的文件，但您可以根据应用程序的需求存储文件。

In [ ]:
import os

dict_o_ntwks_files = rf.io.read_all(
    files=[os.path.join(rf.data.pwd, test_file) for test_file in ['ntwk1.s2p', 'ntwk2.s2p']]
)
dict_o_ntwks_files

## 其他参数本教程重点介绍散射参数，但也可以使用其他网络表示方法。可以通过参数 `Network.z` 和 `Network.y` 分别访问阻抗和导纳参数。还可以使用复数参数的标量分量，例如 `Network.z_re`、`Network.z_im`，以及绘图方法。其他参数仅适用于 2 端口网络，例如波级联参数 (`Network.t`) 和 ABCD 参数 (`Network.a`) 或混合参数 (`Network.h`)。

In [ ]:
ring_slot.z[:3,...]

In [ ]:
ring_slot.plot_z_im(m=1,n=0)

## References[1] Compton, R.C.; , "Perspectives in microwave circuit analysis," Circuits and Systems, 1989., Proceedings of the 32nd Midwest Symposium on , vol., no., pp.716-718 vol.2, 14-16 Aug 1989. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=101955&isnumber=3167